# NaN Handling

Real-world time series have gaps: failed sensors, market halts, missing ticks.
screamer provides two levels of NaN management.  **Shape-preserving** tools
(`FillNa`, `Ffill`, `RollingMean` with the default "ignore" policy) keep the
original length and index intact — suitable when you need aligned arrays.
**Cardinality-changing** tools (`dropna`) remove the NaN events entirely,
producing a shorter but clean stream — useful before feeding data to a model
or aggregation pipeline.

Note: `bfill` (backward fill) does not exist in screamer by design — it would
require looking at future values, which violates causality.

In [ ]:
import numpy as np
from screamer import RollingMean, FillNa, Ffill
from screamer import dropna

x = np.arange(10.0)
x[3] = np.nan
x[7] = np.nan     # two gaps at indices 3 and 7

print("input :", x)

## The "ignore" NaN policy

`RollingMean` (and most rolling / EW estimators) default to the `"ignore"`
NaN policy: a NaN input is **skipped from the internal state** — it does not
corrupt the window — but a NaN is emitted at that index so the output array
stays aligned with the input.

In [ ]:
ignore = RollingMean(3)(x)   # "ignore": NaN skipped from state, NaN emitted at gap

print("ignore:", np.round(ignore, 2))
# indices 0,1 are still in warmup (strict policy, window=3)
# index 3 emits NaN; the window resumes cleanly from index 4 onward

## Shape-preserving gap filling: `FillNa` and `Ffill`

`FillNa(v)` replaces every NaN with the constant `v`.
`Ffill()` (forward-fill) carries the most recent valid value forward —
the standard choice for tick data between trades.

In [ ]:
filled = FillNa(0.0)(x)        # replace NaN with 0
ff     = Ffill()(x)            # carry last valid value forward

print("ffill :", ff)
print("filled:", filled)

# shape is preserved in both cases
assert filled.shape == x.shape
assert ff.shape == x.shape

## Cardinality-changing: `dropna`

`dropna` takes `(keys, values)` — a timed-event pair — and returns the
subset where no value is NaN.  The result is shorter than the input: the
NaN events are gone entirely.

In [ ]:
keys = np.arange(x.size, dtype=np.int64)
dk, dv = dropna(keys, x)

assert dv.size == x.size - 2 and not np.isnan(dv).any()
print("dropped:", dv)
print("keys   :", dk)

## Recovery after a gap

The "ignore" policy means rolling estimators recover gracefully: after a
NaN event the window continues with the valid samples that remain.

In [ ]:
# observe what happens around the gap at index 7
print("\nAround the gap at index 7:")
for i in range(5, 10):
    print(f"  x[{i}]={x[i]:.0f}  ignore[{i}]={ignore[i]:.2f}" if not np.isnan(x[i])
          else f"  x[{i}]=NaN  ignore[{i}]={ignore[i]}")

**Takeaways**

- `FillNa` and `Ffill` are shape-preserving: output length equals input length.
- `dropna` is cardinality-changing: output length equals the number of valid events.
- Rolling/EW estimators use the `"ignore"` policy by default: NaN inputs are
  skipped from state updates and NaN is emitted at that index.
- `bfill` does not exist — backward fill requires future data, which breaks causality.